# Второй проход OCR по ИД: целые страницы бланков актов и реестров

1. Ядро: Colab, среда **T4 GPU** — так же, как для полос.
2. Ячейка 2: вставь ссылку Google Drive на `pages_bundle.zip` (доступ «все, у кого есть ссылка»).
3. Запусти ячейки 1–3, потом **порции по очереди**. Порция — до 150 страниц, около 5–10 минут; в конце она
   сама печатает свой результат. По первой порции видно настоящую скорость (с/стр).
4. **После каждой порции сохраняй ноутбук (Ctrl+S).** Забрать готовые порции на компьютере:
   `python colab/fetch_pages.py` — можно после каждой порции.

Если связь с Colab оборвалась: подключись заново, запусти ячейки 2 и 3 и продолжи с первой порции, у которой
нет строки «порция готова». Готовые порции перезапускать не нужно.
Если ноутбук поменялся на диске — перезагрузи его в VS Code, иначе выполнится старая версия ячеек.

In [3]:
# 1. Диагностика: есть ли GPU
import shutil, subprocess, sys
print("python", sys.version.split()[0])
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)
else:
    print("GPU НЕТ: ноутбук подключён к среде Colab без видеокарты. Смени ядро на новый сервер Colab с T4 GPU "
          "(выбор ядра → Colab → новый сервер → GPU → T4). Если T4 не дают — исчерпан бесплатный лимит GPU, "
          "он восстанавливается через несколько часов.")

python 3.13.15
GPU НЕТ: ноутбук подключён к среде Colab без видеокарты. Смени ядро на новый сервер Colab с T4 GPU (выбор ядра → Colab → новый сервер → GPU → T4). Если T4 не дают — исчерпан бесплатный лимит GPU, он восстанавливается через несколько часов.


In [ ]:
# 2. Данные: pages_bundle.zip (чёрно-белые PNG страниц в 200 dpi + pages_index.jsonl)
import json, os, re, shutil, subprocess, zipfile
BUNDLE = ""   # ссылка «Поделиться» на Google Drive или только ID из неё

NAME = "pages_bundle.zip"

def is_pages(path):
    # тот ли это архив: в pages_bundle.zip есть pages_index.jsonl, в strips_bundle.zip — index.jsonl
    return zipfile.is_zipfile(path) and "pages_index.jsonl" in zipfile.ZipFile(path).namelist()

if os.path.exists(NAME) and not is_pages(NAME):
    os.remove(NAME)                      # неудачная загрузка или не тот архив
    shutil.rmtree("pages_bundle", ignore_errors=True)
if not os.path.exists(NAME):
    subprocess.run(["pip", "install", "-q", "gdown"], check=True)
    import gdown
    m = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", BUNDLE)
    file_id = (m.group(1) or m.group(2)) if m else BUNDLE.strip()
    print("Drive ID:", file_id)
    gdown.download(id=file_id, output=NAME, quiet=False)
    if not is_pages(NAME):
        size = os.path.getsize(NAME) / 2**20 if os.path.exists(NAME) else 0
        os.remove(NAME)
        raise SystemExit(f"По ссылке не тот файл ({size:.0f} МБ): нужен pages_bundle.zip. "
                         "Загрузи его на Drive, открой доступ по ссылке и вставь новую ссылку в BUNDLE.")
if not os.path.exists("pages_bundle/pages_index.jsonl"):
    shutil.rmtree("pages_bundle", ignore_errors=True)
    zipfile.ZipFile(NAME).extractall("pages_bundle")
index = [json.loads(l) for l in open("pages_bundle/pages_index.jsonl", encoding="utf-8")]
print("страниц:", len(index))

In [ ]:
# 3. EasyOCR на GPU и функция одной порции
import base64, hashlib, io, json, subprocess, time, zipfile
subprocess.run(["pip", "install", "-q", "easyocr"], check=True)
import cv2, easyocr, torch
if not torch.cuda.is_available():
    # на процессоре Colab страница идёт ~30 с, все порции — полдня: без GPU не начинаем
    raise SystemExit("GPU нет — порции не запускаю. Смени ядро на сервер Colab с T4 GPU (см. ячейку 1).")
reader = easyocr.Reader(["ru", "en"], gpu=True, verbose=False)
print("модель загружена, GPU:", torch.cuda.get_device_name(0))

"""Полный OCR одной страницы-картинки — общий код для второго прохода по ИД.

Этот файл целиком вставляется в ноутбук colab_pages.ipynb (build_pages_notebook.py), и его же можно
вызвать локально для проверки. Повторяет ml.pages._ocr_page: страница в 200 dpi режется на плитки
2000 px с перекрытием 200 px (EasyOCR ужимает картинки больше ~2560 px), белые плитки пропускаются,
дубли из зоны перекрытия убираются. Координаты — в пунктах PDF от левого верхнего угла, как в кэше ml.pages.
"""
TILE_PX = 2000
OVERLAP_PX = 200


def _iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def ocr_image(reader, img, dpi=200, batch_size=16):
    """Серая картинка страницы (numpy, h × w) → [[текст, x0, y0, x1, y1, уверенность], ...] в пунктах PDF.

    batch_size — сколько строк распознаётся за раз: на GPU это заметно быстрее, на результат не влияет.
    """
    zoom = dpi / 72
    h, w = img.shape[:2]
    step = TILE_PX - OVERLAP_PX
    tokens = []
    for ty in range(0, h, step):
        for tx in range(0, w, step):
            tile = img[ty:ty + TILE_PX, tx:tx + TILE_PX]
            if tile.size == 0 or tile.min() > 200:          # на плитке нет ничего тёмного
                continue
            for box, text, conf in reader.readtext(tile, batch_size=batch_size):
                xs = [p[0] for p in box]
                ys = [p[1] for p in box]
                tokens.append([text, round((tx + min(xs)) / zoom, 2), round((ty + min(ys)) / zoom, 2),
                               round((tx + max(xs)) / zoom, 2), round((ty + max(ys)) / zoom, 2),
                               round(float(conf), 3)])
    # дубли из зоны перекрытия плиток: оставляем более уверенный
    tokens.sort(key=lambda t: -t[5])
    kept = []
    for t in tokens:
        if all(_iou(t[1:5], k[1:5]) < 0.5 for k in kept):
            kept.append(t)
    return kept


def ocr_row(r):
    img = cv2.imread(f"pages_bundle/{r['img']}", cv2.IMREAD_GRAYSCALE)
    rec = {k: r[k] for k in ("file_id", "page", "kind", "page_w", "page_h", "rotation")}
    rec["tokens"] = [] if img is None else ocr_image(reader, img, dpi=r["dpi"], batch_size=16)
    return rec

def run_portion(start, end):
    rows, lines, t0 = index[start:end], [], time.time()
    for i, r in enumerate(rows, 1):
        lines.append(json.dumps(ocr_row(r), ensure_ascii=False))
        if i % 25 == 0 or i == len(rows):
            rate = (time.time() - t0) / i
            print(f"{i}/{len(rows)}  {rate:.1f} с/стр, осталось ~{rate * (len(rows) - i) / 60:.0f} мин", flush=True)
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED, compresslevel=9) as z:
        z.writestr("results.jsonl", "\n".join(lines) + "\n")
    data = buf.getvalue()
    b64 = base64.b64encode(data).decode()
    print(f"===PAGES_BEGIN part={start} sha256={hashlib.sha256(data).hexdigest()}===")
    print("\n".join(b64[i:i + 4000] for i in range(0, len(b64), 4000)))
    print("===PAGES_END===")
    print(f"порция готова: {len(rows)} страниц за {(time.time() - t0) / 60:.0f} мин. Сохрани ноутбук (Ctrl+S).")


In [ ]:
# Порция 1 — F0205 стр.9 – F0208 стр.56, 152 страниц
run_portion(3, 155)

In [ ]:
# Порция 2 — F0208 стр.86 – F0215 стр.166, 152 страниц
run_portion(155, 307)

In [ ]:
# Порция 3 — F0215 стр.168 – F0217 стр.226, 152 страниц
run_portion(307, 459)

In [ ]:
# Порция 4 — F0217 стр.227 – F0219 стр.285, 152 страниц
run_portion(459, 611)

In [ ]:
# Порция 5 — F0219 стр.286 – F0222 стр.11, 152 страниц
run_portion(611, 763)

In [ ]:
# Порция 6 — F0222 стр.12 – F0223 стр.291, 147 страниц
run_portion(763, 910)